In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress

## Functions 

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def make_test_map(data,hdr,vmin=-0.5,vmax=0.5,cmap='RdBu_r',
                  *args,**kwargs):

    wcs = WCS(hdr)
    fs = 20
    plt.figure(figsize=(40,4))
    plt.subplot(projection=wcs)
    plt.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    plt.title(hdr['OBJECT'],fontsize=fs)
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs)

    gc.collect()

    return

In [ ]:
def make_PI_file(directory,Q_list,U_list,hdr,avg_order='avg_PI',keep_PI=True,savefile=False,
                 *args,**kwargs):

    hdr_PI = hdr[0].copy()
    #print(repr(hdr_PI))
    
    filetype = hdr_PI['OBJECT'][3:]
    print(filetype)
    hdr_PI['OBJECT'] = filetype+'_'+avg_order
    
    if avg_order == 'avg_PI':
        PI = np.nanmean(np.sqrt(np.array(Q_list)**2+np.array(U_list)**2),axis=0)
    if avg_order == 'PI_avg':
        PI = np.sqrt(np.nanmean(np.array(Q_list),axis=0)**2+np.nanmean(np.array(U_list),axis=0)**2)

    if savefile:
        fits.writeto(directory+'PI_'+filetype+'_'+avg_order+'.fits',PI,header=hdr_PI,overwrite=True)

    if keep_PI == False:
        del PI
        PI = 0
    gc.collect()

    return PI, hdr_PI

In [ ]:
def make_kernel(fwhm,hdr):
    
    print('FWHM of kernel: '+str(fwhm)+' arcmin')   
    pixsize = hdr[0]['CDELT2']*60
    print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    print('standard deviation of kernel: '+str(stddev)+' pixels')

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    center = ((kernel.shape[0]-1)/2,(kernel.shape[0]-1)/2)
    circle1 = Circle(center, stddev, fill=False)    
    fig,ax = plt.subplots(1,1,figsize=(3,3))
    plt.imshow(kernel, interpolation='none', origin='lower')
    plt.xlabel('lon [pixels]')
    plt.ylabel('lat [pixels]')
    ax.add_patch(circle1)
    ax.set_aspect('equal')
    plt.show()
    
    return kernel,stddev

In [ ]:
def convolve_data(kernel,stddev,data_list,hdr_list,keep_conv=True,savefile=False,
                  regrid_data=False,pixfactor=1,*args,**kwargs):
    
    data_conv = []
    hdr_conv = []
    descrptn = []
    
    for i in range(0,len(data_list)):
            
        data_conv.append(convolve(data_list[i],kernel))
            
        if regrid_data:
            descrptn.append(hdr_list[i]['OBJECT']+'_conv'+str(int(stddev))+'_regrd')
        else:
            descrptn.append(hdr_list[i]['OBJECT']+'_conv'+str(int(stddev)))           
        print(descrptn[i])
                
        hdr_new = hdr_list[i].copy()
        hdr_new['OBJECT'] = descrptn[i]
        hdr_conv.append(hdr_new)
            
        print('finished convolving array '+str(i+1)+' of '+str(len(data_list)))
    
    if regrid_data:
        data_conv,hdr_conv = regrid_maps(pixfactor,data_conv,hdr_conv)

    if savefile:
        for i in range(0,len(data_list)):
            fits.writeto(directory+descrptn[i]+'.fits',data_conv[i],
                         header=hdr_conv[i],overwrite=True)

    if keep_conv == False:
        del data_conv
        data_conv = 0
    gc.collect()
    
    return data_conv,hdr_conv

In [ ]:
def regrid_maps(pixfactor,data_list,hdr_list):
    
    data_regrid = []
    hdr_regrid = []
    
    for i in range(0,len(data_list)):
        
        hdr_new = hdr_list[i].copy()
        hdr_new['CDELT1'] = hdr_new['CDELT1']*pixfactor
        hdr_new['CDELT2'] = hdr_new['CDELT2']*pixfactor
        hdr_new['NAXIS1'] = int(hdr_new['NAXIS1']/pixfactor)
        hdr_new['NAXIS2'] = int(hdr_new['NAXIS2']/pixfactor)
        hdr_new['CRPIX1'] = int(hdr_new['CRPIX1']/pixfactor)
        hdr_new['CRPIX2'] = int(hdr_new['CRPIX2']/pixfactor)

        hdr_regrid.append(hdr_new)
        data_regrid.append(reproject_interp((data_list[i],hdr_list[i]),hdr_new)[0])
        print('finished regridding array '+str(i+1)+' of '+str(len(data_list)))
            
    return data_regrid, hdr_regrid

In [ ]:
def calc_pol_angles(Q_list,U_list,hdrQ_list,hdrU_list,keep_PA=True,
                    savefile=False,*args,**kwargs):
    
    PA = []
    zz = []
    dtau = []
    descrptn = []
    hdr_PA = []
    
    for i in range(0,len(Q_list)):
        
        descrptn.append('PA_'+hdrQ_list[i]['OBJECT'][1:])
        
        PA.append(0.5*np.arctan2(U_list[i],Q_list[i]))
        
        ## correct angle wrap:
        zz.append(np.exp(2*PA[i]*1j))
        if i > 0:
            dtau.append(np.arctan2((zz[i]*np.conj(zz[i-1])).imag,
                                   (zz[i]*np.conj(zz[i-1])).real)/2.)
            PA[i] = PA[i-1]+dtau[i-1]
            
        hdr_new = hdrQ_list[i].copy()
        hdr_new['OBJECT'] = descrptn[i]
        hdr_PA.append(hdr_new)
        
        print('finished calculating PA for array '+str(i+1)+' of '+str(len(Q_list)))
            
    if savefile:
        for i in range(0,len(PA)):
            fits.writeto(directory+descrptn[i]+'.fits',PA[i],
                         header=hdr_PA[i],overwrite=True)
    
    del zz; del dtau
    if keep_PA == False:
        del PA
        PA = 0
    gc.collect()
    
    return PA, hdr_PA

In [ ]:
def calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=False,*args,**kwargs):
    
    freq = np.array([1406.9,1413.8,1427.4,1434.3])
    lbd2 = ((3e8)/(freq*1e6))**2
    
    RM     = np.empty_like(PA_list[0])
    PAint  = np.empty_like(PA_list[0])
    rvalue = np.empty_like(PA_list[0])
    pvalue = np.empty_like(PA_list[0])
    stderr = np.empty_like(PA_list[0])
    intstd = np.empty_like(PA_list[0])
    
    descrptn_RM = 'RM_'+hdrPA_list[0]['OBJECT'][5:]
    
    for i in range(0,RM.shape[0]):
        print(str(i)+' of '+str(RM.shape[0])+' lat pixels')
        for j in range(0,RM.shape[1]):
            if np.isfinite(PA_list[0][i,j]):
                PA = np.array([PA_list[0][i,j],PA_list[1][i,j],
                               PA_list[2][i,j],PA_list[3][i,j]])
                try:
                    result = linregress(lbd2,PA)
                    RM[i,j]     = result.slope
                    PAint[i,j]  = result.intercept
                    rvalue[i,j] = result.rvalue
                    pvalue[i,j] = result.pvalue
                    stderr[i,j] = result.stderr
                    intstd[i,j] = result.intercept_stderr
                except:
                    pass
                del PA

        gc.collect()
            
    hdr_RM = hdrPA_list[0].copy()
    hdr_RM['OBJECT'] = descrptn_RM
    hdu_list = fits.HDUList()
    
    hdr_RM['EXTNAME'] = 'RM'
    hdu1 = fits.ImageHDU(RM, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PA_INT'
    hdu2 = fits.ImageHDU(PAint, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'RVALUE'
    hdu3 = fits.ImageHDU(rvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PVALUE'
    hdu4 = fits.ImageHDU(pvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'STDERR'
    hdu5 = fits.ImageHDU(stderr, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'INT_STDERR'
    hdu6 = fits.ImageHDU(intstd, header=hdr_RM)

    for hdu_app in [hdu1,hdu2,hdu3,hdu4,hdu5,hdu6]:
        print(hdu_app)
        hdu_list.append(hdu_app)

    if savefile:        
        hdu_list.writeto(directory+descrptn_RM+'.fits',overwrite=True)
    
    if keep_RM == False:
        del RM; del hdu_list; del PAint
        del rvalue; del pvalue; del stderr; del intstd
        RM = 0; hdu_list = 0
    gc.collect()
    
    return RM, hdu_list

## Read in 'raw' CG files, convolve, regrid, calculate PA (Aug 22, 2023)

In [ ]:
%%time
Aug22_attempt = False

if Aug22_attempt:

    directory = '/srv/aordog/cgps_gmims_data/'

    CG_Q_list, hdr_CG_Q_list = read_files_4channels(directory,'Q','CG')
    CG_U_list, hdr_CG_U_list = read_files_4channels(directory,'U','CG')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_CG_Q_list)


    CG_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,CG_Q_list,hdr_CG_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    CG_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,CG_U_list,hdr_CG_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(CG_U_conv_list[0].shape)
    print(CG_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(CG_Q_conv_list,CG_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')


## Start from CG PA and calculate RM along with fitting parameters (Aug 23, 2023)

In [ ]:
%%time
Aug23_attempt = False

if Aug23_attempt:

    directory = '/srv/aordog/cgps_gmims_data/'
    
    PA_list, hdrPA_list = read_files_4channels(directory,'PA_','CG_conv4_regrd')
    print(PA_list[0].shape)

    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Read in 'raw' C files, convolve, regrid, calculate PA and RM (Aug 23, 2023)

In [ ]:
%%time
Aug23_C = False

if Aug23_C:

    directory = '/srv/aordog/cgps_gmims_data/'

    C_Q_list, hdr_C_Q_list = read_files_4channels(directory,'Q','C')
    C_U_list, hdr_C_U_list = read_files_4channels(directory,'U','C')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_C_Q_list)


    C_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,C_Q_list,hdr_C_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    C_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,C_U_list,hdr_C_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(C_U_conv_list[0].shape)
    print(C_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(C_Q_conv_list,C_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')
    
    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Read in 'raw' G files, convolve, regrid, calculate PA and RM (Aug 23, 2023)

In [ ]:
%%time
Aug23_G = False

if Aug23_G:

    directory = '/srv/aordog/cgps_gmims_data/'

    G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','G')
    G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','G')

    fwhm = 3 # arcmin
    kernel,stddev = make_kernel(fwhm,hdr_G_Q_list)


    G_Q_conv_list,hdrQ_conv_list = convolve_data(kernel,stddev,G_Q_list,hdr_G_Q_list,
                                                 keep_conv=True,savefile=True,
                                                 regrid_data=True,pixfactor=2)

    G_U_conv_list,hdrU_conv_list = convolve_data(kernel,stddev,G_U_list,hdr_G_U_list,
                                                  keep_conv=True,savefile=True,
                                                  regrid_data=True,pixfactor=2)

    print(G_U_conv_list[0].shape)
    print(G_Q_conv_list[0].shape)

    PA_list, hdrPA_list = calc_pol_angles(G_Q_conv_list,G_U_conv_list,
                                          hdrQ_conv_list,hdrU_conv_list,
                                          keep_PA=True,savefile=True)
    
    make_test_map(PA_list[0],hdrPA_list[0],vmin=-np.pi/2,vmax=np.pi/2,cmap='twilight')
    
    RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=True)
    
    make_test_map(hdu_list[0].data,hdu_list[0].header,vmin=-300,vmax=300,cmap='RdBu_r')
    make_test_map(np.abs(hdu_list[2].data),hdu_list[2].header,vmin=0.5,vmax=1,cmap='viridis')

## Calculate PI for convolved and gridded data

In [ ]:
%%time
Aug23_PI = True

if Aug23_PI:

    directory = '/srv/aordog/cgps_gmims_data/'

    G_Q_list, hdr_G_Q_list = read_files_4channels(directory,'Q','G_conv4_regrd')
    G_U_list, hdr_G_U_list = read_files_4channels(directory,'U','G_conv4_regrd')
    
    C_Q_list, hdr_C_Q_list = read_files_4channels(directory,'Q','C_conv4_regrd')
    C_U_list, hdr_C_U_list = read_files_4channels(directory,'U','C_conv4_regrd')
    
    CG_Q_list, hdr_CG_Q_list = read_files_4channels(directory,'Q','CG_conv4_regrd')
    CG_U_list, hdr_CG_U_list = read_files_4channels(directory,'U','CG_conv4_regrd')
    
    G_avg_PI, hdr_G_avg_PI = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='avg_PI',keep_PI=True,savefile=True)
    G_PI_avg, hdr_G_PI_avg = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='PI_avg',keep_PI=True,savefile=True)
    
    C_avg_PI, hdr_C_avg_PI = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='avg_PI',keep_PI=True,savefile=True)
    C_PI_avg, hdr_C_PI_avg = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='PI_avg',keep_PI=True,savefile=True)
    
    CG_avg_PI, hdr_CG_avg_PI = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='avg_PI',keep_PI=True,savefile=True)
    CG_PI_avg, hdr_CG_PI_avg = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='PI_avg',keep_PI=True,savefile=True)
    

In [ ]:

freq = np.array([1406.9,1413.8,1427.4,1434.3])
lbd2 = ((3e8)/(freq*1e6))**2

print((lbd2[0]-lbd2[1])*100*180/np.pi)

print((2.5*np.pi/180)/(lbd2[0]-lbd2[1]))

In [ ]:
print(500*lbd2)
print(530*lbd2)